# Projet Économétrie Qualitative — Modélisation des notations Yuka
## Parties 1 : Présentation · Statistiques descriptives · Analyse exploratoire

**Auteure :** Palla Gaye  
**Groupe :** Palla Gueye · Havar Baskara · Mouhammad Gaye  
**Master 1 BIDABI — Université Sorbonne Paris Nord**  
**Cours : Économétrie des données qualitatives**

---

## 0. Imports et configuration

On importe les librairies nécessaires et on configure les chemins dynamiques pour que le notebook fonctionne depuis n'importe quel emplacement dans le repo.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, chi2_contingency

import warnings
warnings.filterwarnings("ignore")

# Chemins dynamiques
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(BASE_DIR, "data")):
    BASE_DIR = os.getcwd()
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# Style graphique
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120

print(f"✅ Imports OK")
print(f"📁 BASE_DIR    : {BASE_DIR}")
print(f"📁 FIGURES_DIR : {FIGURES_DIR}")

---
## Partie 1 — Présentation des données

### Objectif
Avant toute analyse, il est indispensable de comprendre la structure du dataset : nombre d'observations, nature des variables, présence de valeurs manquantes, et distribution de la variable à expliquer.

### 1.1 Chargement des données

In [ ]:
df = pd.read_csv(os.path.join(BASE_DIR, "data", "jeu_donnees_yuka.csv"),
                 sep=None, engine="python")
df = df.drop(columns=["id_produit"])

print(f"Nombre d'observations : {df.shape[0]}")
print(f"Nombre de variables   : {df.shape[1]}")
df.head()

### 1.2 Types des variables

In [ ]:
print("Types des variables :")
print(df.dtypes)

**Interprétation :**

- `score_yuka_ordonne` est notre **variable dépendante ordonnée** (1=Mauvais, 2=Médiocre, 3=Bon, 4=Excellent). Son caractère ordonné justifie l'usage d'un **modèle Logit Ordonné** plutôt qu'une régression linéaire ou un logit binaire.
- Les **variables quantitatives** (calories, sucres, etc.) mesurent la composition nutritionnelle des produits.
- Les **variables qualitatives binaires** (bio, ultra_transforme) capturent des caractéristiques structurelles du produit.

### 1.3 Valeurs manquantes

In [ ]:
print("Valeurs manquantes par variable :")
display(df.isnull().sum().to_frame("Nulls"))
print("\n→ Aucune valeur manquante : le dataset est complet, aucun traitement d'imputation n'est nécessaire.")

### 1.4 Distribution de la variable dépendante

In [ ]:
dist_y = df["score_yuka_ordonne"].value_counts().sort_index()
labels = {1: "Mauvais", 2: "Médiocre", 3: "Bon", 4: "Excellent"}

print("Distribution de score_yuka_ordonne :")
for k, v in dist_y.items():
    print(f"  {k} — {labels[k]:10s} : {v:4d} produits ({100*v/len(df):.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#d32f2f", "#f57c00", "#388e3c", "#1565c0"]
bars = ax.bar([labels[k] for k in dist_y.index], dist_y.values, color=colors, edgecolor="white")
for bar, val in zip(bars, dist_y.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{val}\n({100*val/len(df):.1f}%)", ha="center", va="bottom", fontsize=10)
ax.set_title("Distribution de la variable dépendante\nscore_yuka_ordonne", fontsize=13, fontweight="bold")
ax.set_ylabel("Nombre de produits")
ax.set_xlabel("Catégorie Yuka")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig1_distribution_Y.png"))
plt.show()

**Interprétation :**

La majorité des produits (36.8%) sont classés **Mauvais**, ce qui reflète la réalité de l'offre alimentaire industrielle. Seulement 13.3% atteignent le score **Excellent**. Cette distribution déséquilibrée est un fait stylisé important à mentionner avant l'estimation du modèle.

---
## Partie 2 — Statistiques descriptives

In [ ]:
vars_quanti = ["calories_100g", "sucres_100g", "graisses_saturees_100g",
               "sel_100g", "fibres_100g", "proteines_100g", "nb_additifs"]
vars_quali  = ["bio", "ultra_transforme"]

### 2.1 Variables quantitatives — Statistiques descriptives

In [ ]:
stats_desc = df[vars_quanti].agg(["mean", "std", "min",
                                   lambda x: x.quantile(0.25),
                                   "median",
                                   lambda x: x.quantile(0.75),
                                   "max"])
stats_desc.index = ["Moyenne", "Écart-type", "Min", "Q1", "Médiane", "Q3", "Max"]
display(stats_desc.round(3))

**Interprétation économique :**

- **Sucres :** l'OMS recommande moins de 25g/jour — la moyenne observée est déjà préoccupante.
- **Sel :** niveau cohérent avec une consommation industrielle souvent excessive en sodium.
- **Graisses saturées :** associées aux risques cardiovasculaires.
- **Fibres :** bien en dessous des apports journaliers recommandés (25g/jour).
- **Additifs :** indicateur clé de la transformation industrielle.

→ Les variables *néfastes* (sucres, sel, graisses, additifs) devraient avoir un **effet négatif** sur le score Yuka, et les variables *bénéfiques* (fibres, protéines) un **effet positif**.

### 2.2 Histogrammes des variables quantitatives

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, var in enumerate(vars_quanti):
    axes[i].hist(df[var], bins=30, color="#42a5f5", edgecolor="white", alpha=0.85)
    axes[i].set_title(var, fontsize=10, fontweight="bold")
    axes[i].set_ylabel("Fréquence")
for j in range(len(vars_quanti), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Distributions des variables quantitatives", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig2_histogrammes_quanti.png"))
plt.show()

### 2.3 Variables qualitatives — Statistiques descriptives

In [ ]:
for var in vars_quali:
    freq = df[var].value_counts()
    print(f"{var} :")
    print(f"  0 (Non) : {freq.get(0,0):4d} produits ({100*freq.get(0,0)/len(df):.1f}%)")
    print(f"  1 (Oui) : {freq.get(1,0):4d} produits ({100*freq.get(1,0)/len(df):.1f}%)")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, var in enumerate(vars_quali):
    freq = df[var].value_counts().sort_index()
    axes[i].bar(["Non (0)", "Oui (1)"], freq.values, color=["#ef9a9a", "#66bb6a"], edgecolor="white")
    for j, v in enumerate(freq.values):
        axes[i].text(j, v + 5, f"{v} ({100*v/len(df):.1f}%)", ha="center", fontsize=10)
    axes[i].set_title(var, fontsize=11, fontweight="bold")
    axes[i].set_ylabel("Nombre de produits")
plt.suptitle("Distribution des variables qualitatives", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig3_barplots_quali.png"))
plt.show()

**Interprétation :**

- **Bio :** seulement ~19% des produits sont biologiques. Ces produits tendent à avoir moins d'additifs et de meilleurs profils nutritionnels.
- **Ultra-transformé :** ~37% des produits sont ultra-transformés (classification NOVA). Ce taux élevé reflète la structure de l'offre alimentaire moderne.

### 2.4 Matrice de corrélation — Analyse de la multicolinéarité

In [ ]:
corr_matrix = df[vars_quanti].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Matrice de corrélation\n(variables quantitatives)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig4_correlation.png"))
plt.show()

In [ ]:
# Seuil conservateur de 0.5 (plus strict que le 0.7/0.8 souvent utilisé en économétrie)
seuil = 0.5
paires_fortes = []
for i in range(len(vars_quanti)):
    for j in range(i+1, len(vars_quanti)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > seuil:
            paires_fortes.append((vars_quanti[i], vars_quanti[j], r))

if paires_fortes:
    print(f"Corrélations > {seuil} (risque de multicolinéarité) :")
    for v1, v2, r in paires_fortes:
        print(f"  {v1} — {v2} : r = {r:.3f}")
else:
    print(f"Aucune corrélation > {seuil} détectée.")
    print("→ On ne soupçonne pas de multicolinéarité forte, à confirmer lors de l'estimation.")
    print("  Toutes les variables peuvent être incluses en première approche.")

### 2.5 Tests de moyenne (ANOVA) — Variables quantitatives vs Y

**Hypothèses :**
- **H₀ :** la moyenne de X est identique dans toutes les catégories de Y
- **H₁ :** au moins une catégorie a une moyenne différente

Si p-value < 0.05, on rejette H₀ → la variable discrimine bien les catégories de score Yuka → pertinente pour le modèle.

In [ ]:
groupes_y = [df[df["score_yuka_ordonne"] == k] for k in [1, 2, 3, 4]]
vars_sig_anova = []

resultats_anova = []
for var in vars_quanti:
    groupes = [g[var].values for g in groupes_y]
    f_stat, p_val = f_oneway(*groupes)
    sig = "✅ OUI" if p_val < 0.05 else "❌ NON"
    if p_val < 0.05:
        vars_sig_anova.append(var)
    resultats_anova.append({"Variable": var, "F-stat": round(f_stat, 3), "p-value": round(p_val, 4), "Significatif ?": sig})

display(pd.DataFrame(resultats_anova).set_index("Variable"))

**Interprétation ANOVA :**

Pour chaque variable significative, on rejette H₀ : la moyenne de X diffère selon la catégorie Yuka. Ces variables discriminent bien les produits selon leur score.

→ Toutes les variables quantitatives sont significatives au seuil de 5%.  
→ Ces résultats suggèrent des différences de moyennes entre classes, **sans préjuger de la forme fonctionnelle du modèle**.

### 2.6 Tests Chi² — Variables qualitatives vs Y

**Hypothèses :**
- **H₀ :** X et Y sont indépendantes
- **H₁ :** X et Y sont liées

Si p-value < 0.05, on rejette H₀ → la variable qualitative est liée au score Yuka.

In [ ]:
resultats_chi2 = []
for var in vars_quali:
    table = pd.crosstab(df[var], df["score_yuka_ordonne"])
    chi2, p, dof, expected = chi2_contingency(table)
    sig = "✅ OUI" if p < 0.05 else "❌ NON"
    resultats_chi2.append({"Variable": var, "Chi²": round(chi2, 3), "p-value": round(p, 4), "Significatif ?": sig})

display(pd.DataFrame(resultats_chi2).set_index("Variable"))

In [ ]:
# Tableaux croisés (fréquences en %)
for var in vars_quali:
    table = pd.crosstab(df[var], df["score_yuka_ordonne"],
                        rownames=[var], colnames=["Score Yuka"],
                        normalize="index") * 100
    table.index = ["Non (0)", "Oui (1)"]
    table.columns = ["Mauvais", "Médiocre", "Bon", "Excellent"]
    print(f"\n{var} — répartition par score Yuka (%) :")
    display(table.round(1))

**Interprétation Chi² :**

On rejette H₀ pour `bio` et `ultra_transforme` : ces deux variables sont **significativement liées** au score Yuka.
- Un produit bio tend à obtenir un **meilleur** score.
- Un produit ultra-transformé tend à obtenir un **moins bon** score — cohérent avec la littérature sur la qualité nutritionnelle.

→ Les deux variables seront incluses dans le modèle.

---
## Partie 3 — Analyse exploratoire approfondie

### 3.1 Boxplots : variables quantitatives par catégorie Yuka

Ces graphiques permettent de visualiser directement comment chaque variable se distribue selon le score Yuka. C'est la preuve visuelle des résultats ANOVA.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
colors_box = ["#d32f2f", "#f57c00", "#388e3c", "#1565c0"]

for i, var in enumerate(vars_quanti):
    sns.boxplot(data=df, x="score_yuka_ordonne", y=var, palette=colors_box, ax=axes[i])
    axes[i].set_title(var, fontsize=10, fontweight="bold")
    axes[i].set_xlabel("Score Yuka (1=Mauvais → 4=Excellent)")
    axes[i].set_ylabel("")
    axes[i].set_xticklabels(["Mauvais", "Médiocre", "Bon", "Excellent"], fontsize=8)

for j in range(len(vars_quanti), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots : variables quantitatives par catégorie Yuka", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig5_boxplots_par_categorie.png"))
plt.show()

### 3.2 Moyennes par catégorie Yuka

In [ ]:
moyennes = df.groupby("score_yuka_ordonne")[vars_quanti].mean()
moyennes.index = ["Mauvais", "Médiocre", "Bon", "Excellent"]
display(moyennes.round(3))

In [ ]:
print("Lecture économique des tendances :\n")
for var in vars_quanti:
    vals = moyennes[var].values
    tendance = "↘ décroissante" if vals[-1] < vals[0] else "↗ croissante"
    if var in ["fibres_100g", "proteines_100g"]:
        effet = "effet POSITIF attendu"
    elif var in ["sucres_100g", "sel_100g", "graisses_saturees_100g", "nb_additifs"]:
        effet = "effet NÉGATIF attendu"
    elif var == "calories_100g":
        effet = "effet ambigu ou faible"
    else:
        effet = "à vérifier"
    print(f"  {var:<35} tendance {tendance} → {effet}")

**Interprétation :**

- Les produits classés **Mauvais** ont en moyenne plus de sucres, sel, graisses saturées et additifs — tous reconnus comme facteurs de risque pour la santé.
- Les produits **Excellent** ont plus de fibres et de protéines, associés à une meilleure qualité nutritionnelle.
- Cette analyse confirme nos hypothèses économiques et valide l'inclusion de toutes les variables dans le modèle.

### 3.3 Proportion bio et ultra-transformé par catégorie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, var in enumerate(vars_quali):
    prop = df.groupby("score_yuka_ordonne")[var].mean() * 100
    axes[i].bar(["Mauvais", "Médiocre", "Bon", "Excellent"],
                prop.values, color=["#d32f2f", "#f57c00", "#388e3c", "#1565c0"], edgecolor="white")
    for j, v in enumerate(prop.values):
        axes[i].text(j, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)
    axes[i].set_title(f"Proportion '{var}' par catégorie Yuka", fontsize=11, fontweight="bold")
    axes[i].set_ylabel("% de produits")
    axes[i].set_xlabel("Catégorie Yuka")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig6_quali_par_categorie.png"))
plt.show()

**Interprétation :**

- **bio :** la proportion augmente de ~13.6% (Mauvais) à ~30.8% (Excellent). Être bio est associé à de meilleures pratiques de production et moins d'intrants chimiques.
- **ultra_transforme :** la proportion chute de ~53.5% (Mauvais) à ~14.3% (Excellent). La transformation industrielle intensive dégrade la qualité nutritionnelle.

---
## Synthèse — Hypothèses économiques et variables retenues

Cette analyse exploratoire permet d'éviter un modèle mal spécifié.

In [ ]:
print("Variables à effet NÉGATIF attendu sur le score Yuka :")
neg = ["sucres_100g", "sel_100g", "graisses_saturees_100g", "nb_additifs", "ultra_transforme"]
for v in neg:
    print(f"  - {v}")

print("\nVariables à effet POSITIF attendu sur le score Yuka :")
pos = ["fibres_100g", "proteines_100g", "bio"]
for v in pos:
    print(f"  - {v}")

print("\nVariables à effet ambigu :")
print("  - calories_100g : significatif statistiquement mais tendance peu claire économiquement")

print(f"\nMulticolinéarité : corrélation max = {corr_matrix.abs().where(~np.eye(len(vars_quanti),dtype=bool)).max().max():.2f}")
print("→ Aucune corrélation élevée (seuil conservateur 0.5) — à confirmer lors de l'estimation.")
print("\n→ Toutes les variables sont retenues pour le modèle Logit Ordonné.")

**Conclusion de l'analyse exploratoire :**

Les statistiques descriptives nous permettent d'identifier les variables pertinentes avant l'estimation économétrique. Cette analyse exploratoire permet d'éviter un modèle mal spécifié. L'ensemble des variables présente une relation significative avec le score Yuka, ce qui justifie leur inclusion dans le modèle Logit Ordonné.

**Limite à garder en tête :** les résultats restent descriptifs et peuvent être influencés par des variables omises (catégorie de produit, marque, prix). L'estimation économétrique permettra de contrôler ces effets partiellement.

---
*✅ Parties 1, 2, 3 terminées — Dataset prêt pour l'estimation (Havar) et l'évaluation (Mouhammad)*